# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JoelShaji-06/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Binary Classification (or Scoring)

**Why:** The goal is to evaluate candidate content/keyword pairs and predict whether they will achieve a top 10 search engine ranking (`1`) or not (`0`). A classification model outputs a probability score between 0 and 1, allowing us to filter out low-value content ideas before spending time and resources drafting them.

In [6]:
# Quick check of task type framing
task_type = "Binary Classification"
print(f"ML Task Type set to: {task_type}")

ML Task Type set to: Binary Classification


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Predicted Target / Proxy:** `is_top_10` (Binary indicator: 1 if SERP rank <= 10, 0 otherwise)

**Label Source:** Observed outcome (historical search engine ranking position data collected from rank-tracking APIs or search logs).

**Why this Target:** Direct business goals like revenue or conversions are too noisy and delayed. Ranking on page one (top 10) is an observed, quantifiable proxy for high organic search traffic and user intent.

In [7]:
# Quick code check for target variable definition
sample_ranks = [4, 18, 25, 2, 9]
target_is_top_10 = [1 if rank <= 10 else 0 for rank in sample_ranks]

print("Observed Ranks:", sample_ranks)
print("Target Labels (is_top_10):", target_is_top_10)

Observed Ranks: [4, 18, 25, 2, 9]
Target Labels (is_top_10): [1, 0, 0, 1, 1]


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Success Metric:** **ROC-AUC** (Target score: **>= 0.75**)

**Why this Metric:** ROC-AUC measures how effectively the model ranks positive instances (content that hits top 10) higher than negative instances across all thresholds, regardless of class imbalance.

**Secondary Precision Metric:** **Precision@Top 20%** (Target: **>= 70%**). When the team acts on the model's top recommendations, 7 out of 10 recommended articles should successfully reach page one.

In [8]:
# Metric baseline check simulation
from sklearn.metrics import roc_auc_score

# Simulated ground truth and predicted probability scores
y_true = [1, 0, 0, 1, 0, 1, 1, 0]
y_scores = [0.85, 0.20, 0.35, 0.90, 0.15, 0.70, 0.60, 0.40]

auc_score = roc_auc_score(y_true, y_scores)
print(f"Simulated Model ROC-AUC Score: {auc_score:.2f}")
print(f"Target Threshold Met (>= 0.75): {auc_score >= 0.75}")

Simulated Model ROC-AUC Score: 1.00
Target Threshold Met (>= 0.75): True


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** **One row = One Content Item / Target Keyword pair.**

**Features Included:**
* `word_count`: Total word length of the content piece.
* `readability_score`: Standardized readability score.
* `domain_authority`: Domain strength of the publishing site.
* `actual_serp_rank`: Observed organic search position.
* `target_is_top_10`: Binary label (1 if rank <= 10, else 0).

In [9]:
import pandas as pd

# Load sample slice representing the unit of analysis
# One row = One Content Item / Keyword pair

data = {
    'content_id': [101, 102, 103, 104, 105],
    'keyword': ['ai workflow audit', 'claude prompt engineering', 'python ml framing', 'esp32 display tutorial', 'digital logic design'],
    'word_count': [1450, 2200, 850, 1900, 1100],
    'readability_score': [68.5, 74.2, 82.0, 61.3, 59.8],
    'domain_authority': [45, 52, 38, 41, 45],
    'actual_serp_rank': [4, 18, 25, 2, 12]
}

df = pd.DataFrame(data)

# Create the explicit target column
df['target_is_top_10'] = (df['actual_serp_rank'] <= 10).astype(int)

# Display the unit of analysis dataframe
df

,content_id,keyword,word_count,readability_score,domain_authority,actual_serp_rank,target_is_top_10
0,101,ai workflow audit,1450,68.5,45,4,1
1,102,claude prompt engineering,2200,74.2,52,18,0
2,103,python ml framing,850,82.0,38,25,0
3,104,esp32 display tutorial,1900,61.3,41,2,1
4,105,digital logic design,1100,59.8,45,12,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML Beats a Fixed Rule (`if/else` statement):**

1. **Non-Linear Interactions:** Search ranking performance depends on complex, multi-variable interactions (e.g., word count, readability, domain authority, and keyword competitiveness). A simple heuristic rule like `if word_count > 1500 and domain_authority > 40` fails because high domain authority can compensate for shorter content, or niche keywords might require different structural depth.

2. **Algorithm and Intent Drift:** Search engine algorithms and competitor strategies evolve continuously. Fixed `if/else` conditions quickly become obsolete, whereas an ML model can be retrained on recent SERP data to update feature weights automatically.

In [10]:
import pandas as pd

# Demonstrating why a hardcoded rule creates false positives/negatives
data = {
    'content_id': [101, 102, 103, 104],
    'word_count': [1200, 1800, 2100, 950],
    'domain_authority': [55, 30, 25, 60],
    'actual_target_is_top_10': [1, 0, 0, 1]
}

df = pd.DataFrame(data)

# Hardcoded Fixed Rule: "If word count > 1500, predict top 10"
df['fixed_rule_pred'] = (df['word_count'] > 1500).astype(int)

# Rule Error Check
df['rule_correct'] = (df['fixed_rule_pred'] == df['actual_target_is_top_10'])
accuracy = df['rule_correct'].mean()

print(df[['content_id', 'word_count', 'domain_authority', 'actual_target_is_top_10', 'fixed_rule_pred']])
print(f"\nFixed Rule Accuracy: {accuracy * 100:.1f}% (Fails due to non-linear feature interactions)")

   content_id  word_count  domain_authority  actual_target_is_top_10  \
0         101        1200                55                        1   
1         102        1800                30                        0   
2         103        2100                25                        0   
3         104         950                60                        1   

   fixed_rule_pred  
0                0  
1                1  
2                1  
3                0  

Fixed Rule Accuracy: 0.0% (Fails due to non-linear feature interactions)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.